# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the <b>A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya</b> using the `mlcroissant` library.

### Dataset Source
The dataset is described as a Croissant schema and is accessible via the following URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

This dataset provides detailed survey data related to mental health indicators in Kilifi County, Kenya, including demographics and mental health assessment scores (GAD-7, PHQ-9, PSQ).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print overview
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}\n")
print("Keywords:", metadata.keywords)

## 2. Data Overview
We will review the available record sets, fields, and their `@id` values. This helps us understand the dataset's structure and reference entities by their unique identifiers.

In [ ]:
# Get a list of available record sets
record_sets_list = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets_list:
    print(f"  Name: {rs.name}, @id: {rs.id}")

# Explore fields for each record set
for rs in record_sets_list:
    print(f"\nFields for RecordSet @id={rs.id} ({rs.name}):")
    for field in rs.fields:
        print(f"    {field.name} (@id: {field.id}) -- DataType: {field.data_type}")

# Let's print a few sample records for the first record set
main_record_set = record_sets_list[0].id
print(f"\nSample records from RecordSet @id={main_record_set}:")
count = 0
for x in dataset.records(record_set=main_record_set):
    print(x)
    count += 1
    if count >= 3:
        break

## 3. Data Extraction
Let's extract data from each record set into pandas DataFrames for subsequent analysis. We will reference all record sets using their `@id` field, and store each as a separate DataFrame.

In [ ]:
# Collect all record set @ids
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, display columns from the main record set
print("DataFrame columns from main record set:")
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Now we'll apply common EDA steps:
- Filtering records based on specific criteria
- Normalizing numeric fields
- Grouping records by key attributes

We reference all fields and columns using their `@id` values. Let's choose a numeric field (e.g., GAD-7 Score) and a group field (e.g., gender or age group). Please see below the field IDs for the main record set.

In [ ]:
# Display all columns and their field @id for main record set
print("RecordSet Field @ids for main record set:")
for f in dataset.record_sets[0].fields:
    print(f"{f.name}: {f.id}")

In [ ]:
# Example EDA: Filter by GAD-7 score (assuming this field exists, update field_id if needed)
record_set_id = main_record_set
# Use the correct field @id for GAD-7 total score
numeric_field_id = None

# Find the field @id for GAD-7 total score
for f in dataset.record_sets[0].fields:
    if "GAD-7" in f.name and ("score" in f.name.lower() or "total" in f.name.lower()):
        numeric_field_id = f.id
        print(f"Using numeric field: {f.name} (@id: {numeric_field_id})")
        break

if numeric_field_id is None:
    # As fallback, just use the first numeric field
    for f in dataset.record_sets[0].fields:
        if f.data_type in ['Float', 'Integer', 'Number']:
            numeric_field_id = f.id
            print(f"Fallback numeric field: {f.name} (@id: {numeric_field_id})")
            break

df = dataframes[record_set_id]
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (e.g., gender; update field @id as needed)
    group_field_id = None
    for f in dataset.record_sets[0].fields:
        if "gender" in f.name.lower():
            group_field_id = f.id
            print(f"Grouping by: {f.name} (@id: {group_field_id})")
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("Numeric field not found in the DataFrame. Check field IDs and update accordingly.")

## 5. Visualization
Visualizing the distribution of the chosen numeric field (e.g., GAD-7 score) and its relation to a categorical variable (e.g., gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (GAD-7 Score)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id available, plot boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Cannot plot: numeric field or group field missing.")

## 6. Conclusion
This notebook demonstrated how to access and analyze the <b>A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya</b> using the `mlcroissant` library.
- Loaded dataset metadata and records by referencing all entities with their `@id`.
- Performed exploratory analysis and filtering to highlight key attributes (e.g., high GAD-7 score records).
- Normalized numeric fields and grouped data by demographic attributes.
- Visualized distributions and relationships between survey scores and participant demographics.

Next steps could include deeper statistical analysis, machine learning modeling, or continued FAIR data curation.